# Figure 2 — Reliability regime determines model ranking
**A** model × context centered-score heatmap (independent half).
**B** per-context rep_cen distributions (reliability regimes).
**C** pairwise bootstrap CIs for the winner vs runner-up.
Data: five fold-frozen CSVs + `j53_pairwise_ranking_ci.csv`.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RES = os.path.join("..", "..", "results")
FIG = os.path.join("..", "..", "figures")
os.makedirs(FIG, exist_ok=True)
plt.rcParams.update({"figure.dpi": 140, "savefig.dpi": 300,
                     "font.size": 9, "axes.spines.top": False,
                     "axes.spines.right": False})
MODELS = ["lw", "svd20", "moppca5", "nn10", "nn50"]
MODEL_LABELS = {"lw": "LW shrinkage", "svd20": "SVD-20",
                "moppca5": "moPPCA-5", "nn10": "centroid-10",
                "nn50": "centroid-50"}
CONTEXTS = [
    ("K562 essential", "j22_k562_foldfrozen_centroid.csv"),
    ("K562 GWPS", "j22_k562_gwps_foldfrozen_centroid.csv"),
    ("Norman", "j49_norman_foldfrozen.csv"),
    ("RPE1", "j49_rpe1_foldfrozen.csv"),
    ("sci-Plex3", "j49_sciplex3.csv"),
]
def load_ctx(name, fname):
    df = pd.read_csv(os.path.join(RES, fname))
    df["context"] = name
    return df

dfs = [load_ctx(*c) for c in CONTEXTS]
all_df = pd.concat(dfs, ignore_index=True)

# ---- Panel A: heatmap of mean B_cen ----
mat = np.zeros((len(CONTEXTS), len(MODELS)))
for i, (cname, _) in enumerate(CONTEXTS):
    d = all_df[all_df.context == cname]
    for j, m in enumerate(MODELS):
        mat[i, j] = d[f"{m}_B_cen"].mean()
fig, ax = plt.subplots(figsize=(6.4, 3.2))
im = ax.imshow(mat, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(MODELS)), [MODEL_LABELS[m] for m in MODELS])
ax.set_yticks(range(len(CONTEXTS)), [c[0] for c in CONTEXTS])
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i, f"{mat[i,j]:.3f}", ha="center", va="center",
                color="white" if mat[i, j] < mat.max()*0.6 else "black",
                fontsize=8)
ax.set_title("Independent-half centered score: five contexts, five orderings")
fig.colorbar(im, ax=ax, shrink=0.8)
fig.savefig(os.path.join(FIG, "fig2a_ranking_heatmap.png"), bbox_inches="tight")
plt.show()


In [ ]:

# ---- Panel B: rep_cen distributions ----
fig, ax = plt.subplots(figsize=(7.0, 3.0))
for i, (cname, _) in enumerate(CONTEXTS):
    d = all_df[all_df.context == cname].sort_values("rep_cen")
    y = (np.arange(len(d)) + 1) / len(d)
    ax.plot(d["rep_cen"], y, label=cname)
ax.set_xlabel("rep_cen (between-half centered agreement)")
ax.set_ylabel("ECDF")
ax.legend(frameon=False, fontsize=8)
ax.set_title("Replicate-reliability regimes differ by orders of magnitude")
fig.savefig(os.path.join(FIG, "fig2b_repcen_ecdf.png"), bbox_inches="tight")
plt.show()
print(all_df.groupby("context")["rep_cen"].median().round(3))


In [ ]:

# ---- Panel C: winner vs runner-up CIs ----
ci = pd.read_csv(os.path.join(RES, "j53_pairwise_ranking_ci.csv"))
lead = ci.groupby("context").first().reset_index()
fig, ax = plt.subplots(figsize=(6.8, 2.8))
ypos = np.arange(len(lead))
for i, r in lead.iterrows():
    ax.plot([r.ci95_low, r.ci95_high], [i, i], color="0.3", lw=2)
    ax.plot(r.mean_diff, i, "o", color="#b91c1c")
    ax.text(r.ci95_high + 0.004, i, f"{r.better}>{r.worse}", va="center", fontsize=8)
ax.axvline(0, color="0.6", ls=":")
ax.set_yticks(ypos, lead["context"])
ax.set_xlabel("mean paired difference (95% bootstrap CI)")
ax.set_title("Winner vs rank-2: significant in all five contexts")
ax.set_ylim(-0.6, len(lead)-0.2)
fig.savefig(os.path.join(FIG, "fig2c_pairwise_ci.png"), bbox_inches="tight")
plt.show()
lead[["context","better","worse","mean_diff","ci95_low","ci95_high"]]
